# Inference — Hurst Exponent Estimation on Real Financial Data

Applies trained models to S&P 500 and Dow Jones historical price series,
estimates the Hurst exponent over rolling windows, and visualises the results.

## 1. Imports & configuration

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

from utils.scoring import get_trained_model_by_name
from utils.config import load_config


In [ ]:
# --- Inference configuration ---
MODEL_INPUT_DIM = 100  # feature dimension each model was trained on
MODEL_SLIDING_STEP = 1  # step when averaging sub-windows inside a data window

DATA_WINDOW_SIZE = 200  # rolling window applied to the price series
DATA_SLIDING_STEP = 1  # stride between consecutive windows

ANALYSIS_DATE_FROM = "1960-01-01"
ANALYSIS_DATE_TO = "1980-01-01"

# Models highlighted in the comparison plots
HIGHLIGHT_MODELS = [
    "RSEstimator",
    "DFAEstimator",
    "MLPRegressor",
    "RNNRegressor",
    "CNNRegressor",
]

# Colorblind-safe palette (same order as HIGHLIGHT_MODELS)
HIGHLIGHT_COLORS = ["#666666", "#000000", "#7570b3", "#e7298a", "#66a61e"]

# Publication-ready matplotlib defaults
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 11,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "lines.linewidth": 1.2,
    }
)


## 2. Helper functions

In [ ]:
# --- Data loading ---


def load_sp500(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()
    df["Date"] = pd.to_datetime(df["Date"])
    df["Close"] = df["Close"].str.replace(",", "").astype(float)
    return df[["Date", "Close"]]


def load_dow_jones(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    df.columns = ["Date"] + df.columns[1:].tolist()
    df.columns = df.columns.str.strip()
    df["Date"] = pd.to_datetime(df["Date"])
    return df[["Date", "Close"]]


In [ ]:
# --- Feature engineering ---


def build_sliding_window_features(
    data: pd.DataFrame,
    window_size: int,
    step: int = 1,
) -> pd.DataFrame:
    """Slide a normalised window over `data["Close"]` and return a feature matrix.

    Each row holds one Min-Max-scaled window plus the date of its last observation.
    """
    rows = []
    scaler = MinMaxScaler(feature_range=(0, 1))

    for start in range(0, len(data) - window_size + 1, step):
        window = data.iloc[start : start + window_size].copy()
        normalised = scaler.fit_transform(window[["Close"]]).flatten()
        rows.append(
            {
                "Date": window["Date"].iloc[-1],
                **{f"x{i}": v for i, v in enumerate(normalised)},
            }
        )

    return pd.DataFrame(rows)


In [ ]:
# --- Inference ---


def _get_model_label(model) -> str:
    return getattr(model, "name", type(model).__name__)


def predict_with_averaging(
    X: pd.DataFrame,
    model,
    model_input_dim: int,
    step: int = 1,
) -> pd.Series:
    """Return a Series of predictions.

    When the feature matrix is wider than `model_input_dim`, predictions are
    averaged over all sub-windows of the correct width.
    Classical estimators (RS, DFA) are applied directly.
    """
    cls = type(model).__name__.lower()
    is_classical = cls in ("rsestimator", "dfaestimator")

    if is_classical or X.shape[1] <= model_input_dim:
        return pd.Series(model.predict(X), index=X.index, name="prediction")

    n_sub = (X.shape[1] - model_input_dim) // step + 1
    cols = [f"x{i}" for i in range(model_input_dim)]
    sub_preds = []

    for i in range(n_sub):
        start = i * step
        sub_X = X.iloc[:, start : start + model_input_dim].copy()
        sub_X.columns = cols
        sub_preds.append(model.predict(sub_X))

    return pd.DataFrame(sub_preds).mean(axis=0).set_axis(X.index)


def run_inference(
    features: pd.DataFrame,
    models: list,
    model_input_dim: int,
    step: int = 1,
) -> pd.DataFrame:
    """Run all models and return a wide DataFrame (one column per model + Date)."""
    X = features.drop(columns=["Date"])
    results = {"Date": features["Date"].values}

    for model in models:
        label = _get_model_label(model)
        results[label] = predict_with_averaging(X, model, model_input_dim, step).values

    return pd.DataFrame(results)


In [ ]:
# --- Visualisation ---


def plot_price_series(data: pd.DataFrame, ylabel: str = "Closing price (USD)") -> None:
    fig, ax = plt.subplots(figsize=(6.0, 3.8))
    ax.plot(data["Date"], data["Close"], color="black")
    ax.set_xlabel("Date")
    ax.set_ylabel(ylabel)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
    fig.tight_layout()
    plt.show()


def plot_hurst_estimates(
    results: pd.DataFrame,
    cols: list,
    colors: list,
) -> None:
    fig, ax = plt.subplots(figsize=(15, 6))

    for col, color in zip(cols, colors):
        ax.plot(results["Date"], results[col], color=color, label=col)

    ax.set_xlabel("Date")
    ax.set_ylabel("Estimated Hurst exponent")
    ax.tick_params(direction="in", which="both")
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
    fig.tight_layout()
    plt.show()


def plot_correlation_heatmap(results: pd.DataFrame) -> None:
    corr = results.drop(columns=["Date"]).corr()

    fig, ax = plt.subplots(figsize=(10, 10))
    sns.heatmap(
        corr,
        ax=ax,
        cmap="coolwarm",
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        linewidths=0.4,
        cbar_kws={"label": "Pearson correlation coefficient", "shrink": 0.8},
        annot=True,
        fmt=".2f",
        annot_kws={"size": 9},
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    fig.tight_layout()
    plt.show()


## 3. Load models

In [ ]:
config = load_config()

models = [get_trained_model_by_name(name, MODEL_INPUT_DIM) for name in config.models]


## 4. S&P 500

In [ ]:
sp500_raw = load_sp500("data/inference/sp500_old.csv")
plot_price_series(sp500_raw)


In [ ]:
sp500_features = build_sliding_window_features(
    sp500_raw, DATA_WINDOW_SIZE, DATA_SLIDING_STEP
)
sp500_results = run_inference(
    sp500_features, models, MODEL_INPUT_DIM, MODEL_SLIDING_STEP
)

sp500_results.to_csv("inference/sp500.csv", index=False)


In [ ]:
sp500_results = pd.read_csv("inference/sp500.csv", parse_dates=["Date"])
sp500_analysis = sp500_results[
    sp500_results["Date"].between(ANALYSIS_DATE_FROM, ANALYSIS_DATE_TO)
].copy()

sp500_analysis


In [ ]:
plot_hurst_estimates(sp500_analysis, HIGHLIGHT_MODELS, HIGHLIGHT_COLORS)


In [ ]:
plot_correlation_heatmap(sp500_analysis)


## 5. Dow Jones Industrial Average

In [ ]:
dj_raw = load_dow_jones("data/inference/dja.csv")
plot_price_series(dj_raw)


In [ ]:
dj_features = build_sliding_window_features(dj_raw, DATA_WINDOW_SIZE, DATA_SLIDING_STEP)
dj_results = run_inference(dj_features, models, MODEL_INPUT_DIM, MODEL_SLIDING_STEP)

dj_results.to_csv("inference/dj.csv", index=False)


In [ ]:
dj_results = pd.read_csv("inference/dj.csv", parse_dates=["Date"])
dj_analysis = dj_results[
    dj_results["Date"].between(ANALYSIS_DATE_FROM, ANALYSIS_DATE_TO)
].copy()

dj_analysis


In [ ]:
plot_hurst_estimates(dj_analysis, HIGHLIGHT_MODELS, HIGHLIGHT_COLORS)


In [ ]:
plot_correlation_heatmap(dj_analysis)
